# Part 2 — AutoGluon Capabilities Tour

This notebook gives a quick overview of AutoGluon using small, fast examples.

The runnable sections cover:

1. Binary Classification
2. Multiclass Classification
3. Regression
4. Time-Series Forecasting

The remaining AutoGluon capabilities are summarized at the end for revision.

The main pattern to remember is:

**prepare data → fit predictor → inspect leaderboard → predict**


## 1. Setup

### What we saw before
- This is the starting point of the notebook.
- Before training models, AutoGluon and the required Python libraries must be available.

### How this cell is different
- This cell only prepares the environment.
- It does not train any model.

### What this cell does
- Installs only the **AutoGluon Tabular module with LightGBM** instead of the complete AutoGluon package.
- Imports `TabularPredictor`.
- Imports utilities for creating and splitting datasets.
- Defines a fixed random seed.

### What we achieve after running it
- The notebook is ready for quick classification and regression experiments.
- Installation is much lighter than installing every AutoGluon module.

### Remember
For this notebook we only need:

**AutoGluon Tabular + LightGBM**

The core object is:

```python
TabularPredictor
```


In [ ]:
!pip install -q "autogluon.tabular[lightgbm]" "autogluon.timeseries"

import warnings
import tempfile
import os

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.datasets import load_wine

from autogluon.tabular import TabularPredictor
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

WORK = tempfile.mkdtemp(prefix="autogluon_part2_")

print("Setup completed.")
print("Working directory:", WORK)


## 2. Binary Classification — Customer Churn

### What we saw before
- The previous cell only prepared AutoGluon.
- We have not trained a model yet.

### How this cell is different
- This is our first supervised-learning task.
- The target has only two possible values: `0` or `1`.

### What this cell does
- Creates a small customer dataset.
- Splits it into training and test data.
- `TabularPredictor(label="churned")` tells AutoGluon what to predict.
- `eval_metric="roc_auc"` tells AutoGluon how to evaluate the model.
- `hyperparameters={"GBM": {}}` trains only one LightGBM model, which keeps this example fast.
- `predict()` returns class labels.
- `predict_proba()` returns class probabilities.

### What we achieve after running it
- AutoGluon automatically recognizes the task as binary classification.
- We get a model score, predicted classes, and churn probabilities.

### Remember
Binary classification:

**2 possible classes**

`predict()` → class

`predict_proba()` → probability


In [ ]:
n = 500

tenure = rng.integers(1, 73, n)
monthly_charge = rng.uniform(20, 120, n).round(2)
support_calls = rng.poisson(1.2, n)

contract = rng.choice(
    ["month-to-month", "one-year", "two-year"],
    n,
    p=[0.55, 0.25, 0.20]
)

logit = (
    -2.2
    + 1.4 * (contract == "month-to-month")
    - 0.8 * (contract == "two-year")
    + 0.025 * (monthly_charge - 60)
    + 0.40 * support_calls
    - 0.018 * tenure
)

prob = 1 / (1 + np.exp(-logit))
churned = (rng.random(n) < prob).astype(int)

churn = pd.DataFrame({
    "tenure_months": tenure,
    "monthly_charge": monthly_charge,
    "contract": contract,
    "support_calls": support_calls,
    "churned": churned
})

train_c, test_c = train_test_split(
    churn,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=churn["churned"]
)

pred_churn = TabularPredictor(
    label="churned",
    eval_metric="roc_auc",
    path=os.path.join(WORK, "binary"),
    verbosity=0
).fit(
    train_c,
    hyperparameters={"GBM": {}}
)

print("Problem type:", pred_churn.problem_type)
print("Best model:", pred_churn.model_best)

leaderboard_c = pred_churn.leaderboard(
    test_c,
    silent=True
)

display(
    leaderboard_c[
        ["model", "score_test", "score_val"]
    ].head()
)

X_test_c = test_c.drop(columns=["churned"])

churn_pred = pred_churn.predict(X_test_c)
churn_prob = pred_churn.predict_proba(X_test_c)[1]

result_c = X_test_c.head().copy()
result_c["P(churn)"] = churn_prob.head().round(3).values
result_c["predicted"] = churn_pred.head().values
result_c["actual"] = test_c["churned"].head().values

display(result_c)


## 3. Multiclass Classification — Wine Type

### What we saw before
- Binary classification predicted one of two possible outcomes.
- We used `fit()`, `leaderboard()`, `predict()`, and `predict_proba()`.

### How this cell is different
- The target now has **three classes** instead of two.
- The overall AutoGluon workflow remains almost unchanged.

### What this cell does
- Loads the Wine dataset.
- Uses `wine_class` as the target.
- Trains one LightGBM model.
- AutoGluon automatically detects a multiclass problem.
- `predict_proba()` now returns one probability column for each class.

### What we achieve after running it
- We see that the same AutoGluon interface works for targets with more than two categories.

### Remember
Binary classification:

**2 classes**

Multiclass classification:

**3 or more classes**

The workflow stays the same.


In [ ]:
wine = load_wine(as_frame=True)
wine_df = wine.frame.rename(
    columns={"target": "wine_class"}
)

train_m, test_m = train_test_split(
    wine_df,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=wine_df["wine_class"]
)

pred_multi = TabularPredictor(
    label="wine_class",
    path=os.path.join(WORK, "multiclass"),
    verbosity=0
).fit(
    train_m,
    hyperparameters={"GBM": {}}
)

print("Problem type:", pred_multi.problem_type)
print("Classes:", pred_multi.class_labels)
print("Best model:", pred_multi.model_best)

leaderboard_m = pred_multi.leaderboard(
    test_m,
    silent=True
)

display(
    leaderboard_m[
        ["model", "score_test", "score_val"]
    ].head()
)

X_test_m = test_m.drop(columns=["wine_class"])

multi_prob = pred_multi.predict_proba(
    X_test_m
)

print("Probability columns:", list(multi_prob.columns))
display(multi_prob.head())


## 4. Regression — House Price Prediction

### What we saw before
- Binary and multiclass classification predicted categories.
- Their output was a class or class probabilities.

### How this cell is different
- Regression predicts a **continuous numerical value**.
- Here the target is house price.

### What this cell does
- Creates a small synthetic housing dataset.
- Uses `price` as the target.
- `eval_metric="mean_absolute_error"` evaluates the typical prediction error.
- Trains one LightGBM model.
- `predict()` returns a numerical price estimate.

### What we achieve after running it
- AutoGluon automatically recognizes the task as regression.
- We get predicted prices and an easy-to-understand absolute error.

### Remember
Classification predicts:

**a category**

Regression predicts:

**a number**

For MAE:

**smaller error is better.**


In [ ]:
n = 500

sqft = rng.integers(600, 4000, n)
bedrooms = rng.integers(1, 6, n)
age = rng.integers(0, 80, n)

neighborhood = rng.choice(
    ["Riverside", "Downtown", "Suburb", "Hillcrest"],
    n
)

neighborhood_effect = pd.Series(
    neighborhood
).map({
    "Riverside": 70000,
    "Downtown": 100000,
    "Suburb": 30000,
    "Hillcrest": 85000
}).to_numpy()

price = (
    50000
    + sqft * 160
    + bedrooms * 12000
    - age * 1200
    + neighborhood_effect
    + rng.normal(0, 35000, n)
).round(0)

houses = pd.DataFrame({
    "sqft": sqft,
    "bedrooms": bedrooms,
    "age": age,
    "neighborhood": neighborhood,
    "price": price
})

train_r, test_r = train_test_split(
    houses,
    test_size=0.25,
    random_state=RANDOM_STATE
)

pred_reg = TabularPredictor(
    label="price",
    eval_metric="mean_absolute_error",
    path=os.path.join(WORK, "regression"),
    verbosity=0
).fit(
    train_r,
    hyperparameters={"GBM": {}}
)

print("Problem type:", pred_reg.problem_type)
print("Best model:", pred_reg.model_best)

leaderboard_r = pred_reg.leaderboard(
    test_r,
    silent=True
)

display(
    leaderboard_r[
        ["model", "score_test", "score_val"]
    ].head()
)

X_test_r = test_r.drop(columns=["price"])

price_pred = pred_reg.predict(
    X_test_r
)

result_r = test_r[
    ["sqft", "bedrooms", "neighborhood", "price"]
].copy()

result_r["predicted_price"] = price_pred.round(0).values

result_r["absolute_error"] = (
    result_r["predicted_price"]
    - result_r["price"]
).abs()

display(result_r.head())

print(
    "Mean absolute error:",
    round(result_r["absolute_error"].mean(), 2)
)


## 5. Time-Series Forecasting — Predict Future Values

### What we saw before
- Binary classification, multiclass classification, and regression all worked with tabular rows.
- Those tasks predicted a class or a numerical value for each independent row.

### How this cell is different
- Time-series forecasting uses **ordered historical data**.
- The goal is to predict future values from past values.
- To keep this notebook fast, only one simple forecasting model is used.

### What this cell does
- Creates two small daily sales time series.
- Converts them into AutoGluon's `TimeSeriesDataFrame`.
- Keeps the final three days as test data.
- `prediction_length=3` means we forecast the next three days.
- Trains only `SeasonalNaive`, which is lightweight and fast.
- `predict()` returns the future forecast.
- `leaderboard()` evaluates the forecasting model on the held-out time periods.

### What we achieve after running it
- We see that AutoGluon has a separate predictor designed specifically for time-series data.
- We get a three-day forecast for each store without running a heavy model search.

### Remember
Tabular ML:

**rows are usually treated independently**

Time-series forecasting:

**time order matters**

The main objects are:

```python
TimeSeriesDataFrame
TimeSeriesPredictor
```


In [ ]:
# Create two small daily sales series
dates = pd.date_range(
    "2026-01-01",
    periods=30,
    freq="D"
)

frames = []

for store in range(2):
    t = np.arange(len(dates))

    sales = (
        80
        + store * 20
        + 10 * np.sin(2 * np.pi * t / 7)
        + rng.normal(0, 3, len(t))
    ).round()

    frames.append(
        pd.DataFrame({
            "item_id": f"store_{store}",
            "timestamp": dates,
            "target": sales
        })
    )

sales_df = pd.concat(
    frames,
    ignore_index=True
)

ts_data = TimeSeriesDataFrame.from_data_frame(
    sales_df,
    id_column="item_id",
    timestamp_column="timestamp"
)

PREDICTION_LENGTH = 3

train_ts, test_ts = ts_data.train_test_split(
    prediction_length=PREDICTION_LENGTH
)

ts_predictor = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH,
    freq="D",
    eval_metric="MASE",
    path=os.path.join(WORK, "timeseries"),
    verbosity=0
).fit(
    train_ts,
    hyperparameters={
        "SeasonalNaive": {}
    }
)

print("Best forecasting model:", ts_predictor.model_best)

ts_leaderboard = ts_predictor.leaderboard(
    test_ts,
    silent=True
)

display(
    ts_leaderboard[
        ["model", "score_test", "score_val"]
    ]
)

forecast = ts_predictor.predict(
    train_ts
)

print("3-day forecast for store_0:")
display(forecast.loc["store_0"])


# 6. AutoGluon Capability Map

### What we saw before
- We directly executed:
  - binary classification
  - multiclass classification
  - regression
  - time-series forecasting

### How this section is different
- Instead of running more heavy modules, this section summarizes the broader AutoGluon landscape for revision.

### What this section covers
AutoGluon is modular and supports several kinds of machine-learning problems.

| Capability | Main idea |
|---|---|
| Binary classification | Predict one of two classes |
| Multiclass classification | Predict one of three or more classes |
| Regression | Predict a continuous number |
| Quantile regression | Predict ranges/quantiles |
| Time-series forecasting | Predict future values from historical sequences |
| Text + tabular | Combine text with normal table features |
| Images | Train on image data |
| Multimodal | Combine text, images, and tabular data |
| Feature importance | Understand which features influence predictions |
| Model persistence | Save and load trained predictors |

### What we achieve
- We see that AutoGluon is not one algorithm.
- It is an AutoML framework with specialized modules for different data types and tasks.

### Remember
For tabular problems, the core pattern is:

```python
predictor = TabularPredictor(
    label="target"
).fit(train_data)

leaderboard = predictor.leaderboard(test_data)

predictions = predictor.predict(new_data)
```

The most important things we choose are:

**target + evaluation metric + data split**

AutoGluon automates much of the model training and selection.
